# GPT-4o-mini Fine-tuning for Performance Bug Detection

This notebook implements the fine-tuning methodology from Section V.A of the paper.

## Key Details:
- Model: GPT-4o-mini
- Training examples: 392 bugs (80%)
- Validation examples: 98 bugs (20%)
- Training format: buggy code → {category, fixed_code, explanation}
- Expected improvement: 67.3% → 83.7% accuracy

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
from pathlib import Path
import time
from datetime import datetime
import os
from openai import OpenAI
from typing import Dict, List
import matplotlib.pyplot as plt

from config import DATA_DIR, MODEL_CONFIG, MODELS_DIR
from models.fine_tuning import DatasetPreparer, ModelFineTuner, TrainingExample

## 1. Load Training Data

In [ ]:
# Load train/test splits
train_file = DATA_DIR / "train_bugs.json"
test_file = DATA_DIR / "test_bugs.json"

with open(train_file, 'r') as f:
    train_data = json.load(f)
with open(test_file, 'r') as f:
    test_data = json.load(f)

print(f"Training examples: {len(train_data)}")
print(f"Test examples: {len(test_data)}")

# Show category distribution
from collections import Counter
train_categories = Counter([bug['category'] for bug in train_data])
print("\nTraining set categories:")
for cat, count in train_categories.items():
    print(f"  {cat}: {count}")

## 2. Prepare Training Examples

Convert bugs to the format required for OpenAI fine-tuning.

In [ ]:
def create_training_example(bug: Dict) -> Dict:
    """
    Create a training example in OpenAI format.
    Input: buggy code
    Output: JSON with category, fix, and explanation
    """
    
    # Get code from first modified method
    buggy_code = ""
    fixed_code = ""
    
    if isinstance(bug.get('modified_methods'), list) and len(bug['modified_methods']) > 0:
        method = bug['modified_methods'][0]
        if isinstance(method, dict):
            buggy_code = method.get('buggy_code', '')[:1500]  # Limit length
            fixed_code = method.get('fixed_code', '')[:1500]
    
    if not buggy_code:
        return None
    
    # Create explanation based on category
    category = bug['category']
    explanations = {
        'algorithmic_inefficiency': "The code has inefficient algorithmic complexity. The fix optimizes the algorithm for better performance.",
        'memory_usage': "The code creates excessive memory allocations. The fix reduces memory usage through more efficient data structures.",
        'redundant_computation': "The code performs redundant calculations. The fix eliminates repeated computations.",
        'cpu_overhead': "The code has unnecessary CPU overhead. The fix reduces CPU cycles through optimization.",
        'io_inefficiency': "The code performs inefficient I/O operations. The fix improves I/O performance through buffering or batching."
    }
    
    explanation = explanations.get(category, "Performance issue identified and fixed.")
    
    # Create the training example
    system_prompt = """You are an expert at detecting and fixing performance bugs in Java code. 
Analyze the code, identify the performance issue category, provide the fix, and explain the improvement."""
    
    user_prompt = f"""Analyze this Java code for performance issues:
```java
{buggy_code}
```"""
    
    assistant_response = json.dumps({
        "category": category,
        "fixed_code": fixed_code,
        "explanation": explanation,
        "bug_report": f"Performance bug detected: {category.replace('_', ' ')}",
        "performance_impact": "Significant performance improvement"
    }, indent=2)
    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_response}
        ]
    }

In [ ]:
# Create training examples
training_examples = []
validation_examples = []

print("Creating training examples...")
for bug in train_data:
    example = create_training_example(bug)
    if example:
        training_examples.append(example)

print("Creating validation examples...")
for bug in test_data[:20]:  # Use subset for validation
    example = create_training_example(bug)
    if example:
        validation_examples.append(example)

print(f"\nCreated {len(training_examples)} training examples")
print(f"Created {len(validation_examples)} validation examples")

# Show example
if training_examples:
    print("\nExample training instance:")
    print("User prompt (first 200 chars):")
    print(training_examples[0]['messages'][1]['content'][:200])
    print("\nAssistant response (parsed):")
    response = json.loads(training_examples[0]['messages'][2]['content'])
    print(f"Category: {response['category']}")
    print(f"Explanation: {response['explanation'][:100]}...")

## 3. Save Training Files

Save in JSONL format for OpenAI API.

In [ ]:
# Save training file
training_file = DATA_DIR / "fine_tuning" / "training_data.jsonl"
training_file.parent.mkdir(parents=True, exist_ok=True)

with open(training_file, 'w') as f:
    for example in training_examples:
        f.write(json.dumps(example) + '\n')

print(f"Saved training data to {training_file}")
print(f"File size: {training_file.stat().st_size / 1024 / 1024:.2f} MB")

# Save validation file
validation_file = DATA_DIR / "fine_tuning" / "validation_data.jsonl"

with open(validation_file, 'w') as f:
    for example in validation_examples:
        f.write(json.dumps(example) + '\n')

print(f"Saved validation data to {validation_file}")

## 4. Initialize OpenAI Client

Set up connection to OpenAI API for fine-tuning.

In [ ]:
# Initialize OpenAI client
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Please set OPENAI_API_KEY environment variable")
else:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialized")
    
    # Test connection
    try:
        models = client.models.list()
        print(f"Connected to OpenAI. Available models: {len(list(models))}")
    except Exception as e:
        print(f"Connection error: {e}")

## 5. Upload Training Files

In [ ]:
# Upload training file
print("Uploading training file...")
with open(training_file, 'rb') as f:
    training_file_response = client.files.create(
        file=f,
        purpose="fine-tune"
    )

training_file_id = training_file_response.id
print(f"Training file uploaded: {training_file_id}")

# Upload validation file
print("Uploading validation file...")
with open(validation_file, 'rb') as f:
    validation_file_response = client.files.create(
        file=f,
        purpose="fine-tune"
    )

validation_file_id = validation_file_response.id
print(f"Validation file uploaded: {validation_file_id}")

## 6. Create Fine-tuning Job

Configure and start the fine-tuning process.

In [ ]:
# Create fine-tuning job
print("Creating fine-tuning job...")

job_response = client.fine_tuning.jobs.create(
    model="gpt-4o-mini-2024-07-18",  # Latest GPT-4o-mini
    training_file=training_file_id,
    validation_file=validation_file_id,
    hyperparameters={
        "n_epochs": 3,  # As per paper
        "batch_size": 4,
        "learning_rate_multiplier": 0.5
    },
    suffix="performance-bugs"
)

job_id = job_response.id
print(f"Fine-tuning job created: {job_id}")
print(f"Status: {job_response.status}")

# Save job info
job_info = {
    "job_id": job_id,
    "created_at": datetime.now().isoformat(),
    "training_file": training_file_id,
    "validation_file": validation_file_id,
    "model": "gpt-4o-mini",
    "hyperparameters": {
        "n_epochs": 3,
        "batch_size": 4,
        "learning_rate_multiplier": 0.5
    }
}

job_file = MODELS_DIR / "fine_tuning_job.json"
job_file.parent.mkdir(parents=True, exist_ok=True)
with open(job_file, 'w') as f:
    json.dump(job_info, f, indent=2)

print(f"Job info saved to {job_file}")

## 7. Monitor Fine-tuning Progress

In [ ]:
# Monitor job status
print("Monitoring fine-tuning job...")
print("This typically takes 30-60 minutes for our dataset size.")
print("")

start_time = time.time()
status_history = []

while True:
    # Get job status
    job = client.fine_tuning.jobs.retrieve(job_id)
    
    elapsed = (time.time() - start_time) / 60
    status = job.status
    
    # Record status
    status_history.append({
        "time": elapsed,
        "status": status
    })
    
    print(f"[{elapsed:.1f} min] Status: {status}")
    
    if status == "succeeded":
        print(f"\n✅ Fine-tuning completed successfully!")
        print(f"Fine-tuned model: {job.fine_tuned_model}")
        
        # Save model info
        model_info = {
            "model_id": job.fine_tuned_model,
            "job_id": job_id,
            "created_at": datetime.now().isoformat(),
            "training_examples": len(training_examples),
            "validation_examples": len(validation_examples),
            "training_time_minutes": elapsed
        }
        
        model_file = MODELS_DIR / "fine_tuned_model.json"
        with open(model_file, 'w') as f:
            json.dump(model_info, f, indent=2)
        
        print(f"Model info saved to {model_file}")
        break
        
    elif status == "failed":
        print(f"\n❌ Fine-tuning failed!")
        print(f"Error: {job.error}")
        break
    
    elif status == "cancelled":
        print(f"\n⚠️ Fine-tuning was cancelled")
        break
    
    # Wait before next check
    time.sleep(60)  # Check every minute

## 8. Test Fine-tuned Model

Quick test to verify the model is working.

In [ ]:
# Load model info
model_file = MODELS_DIR / "fine_tuned_model.json"
with open(model_file, 'r') as f:
    model_info = json.load(f)

fine_tuned_model = model_info['model_id']
print(f"Testing model: {fine_tuned_model}")

# Test with an example
test_code = """
public void processItems(List<String> items) {
    for (int i = 0; i < items.size(); i++) {
        for (int j = 0; j < items.size(); j++) {
            String result = items.get(i) + items.get(j);
            System.out.println(result);
        }
    }
}
"""

# Get prediction
response = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[
        {"role": "system", "content": "You are an expert at detecting and fixing performance bugs."},
        {"role": "user", "content": f"Analyze this Java code for performance issues:\n```java\n{test_code}\n```"}
    ],
    temperature=0.3,
    max_tokens=500
)

# Parse response
response_text = response.choices[0].message.content
print("Model response:")
print(response_text[:500])

# Try to parse as JSON
try:
    result = json.loads(response_text)
    print("\nParsed result:")
    print(f"Category: {result.get('category')}")
    print(f"Explanation: {result.get('explanation', '')[:200]}")
except:
    print("\n(Response is not valid JSON, but that's okay for testing)")

## 9. Compare Base vs Fine-tuned Model

Quick comparison to show improvement.

In [ ]:
# Test both models on same example
test_examples = test_data[:5]  # Use 5 test examples

base_correct = 0
finetuned_correct = 0

print("Comparing base vs fine-tuned model...\n")

for i, bug in enumerate(test_examples):
    if not bug.get('modified_methods'):
        continue
        
    method = bug['modified_methods'][0]
    if not isinstance(method, dict) or 'buggy_code' not in method:
        continue
        
    buggy_code = method['buggy_code'][:500]
    true_category = bug['category']
    
    prompt = f"Identify the performance bug category in this code:\n```java\n{buggy_code}\n```\nCategories: algorithmic_inefficiency, memory_usage, redundant_computation, cpu_overhead, io_inefficiency"
    
    # Test base model
    base_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=50
    )
    
    # Test fine-tuned model  
    finetuned_response = client.chat.completions.create(
        model=fine_tuned_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=50
    )
    
    base_pred = base_response.choices[0].message.content.lower()
    finetuned_pred = finetuned_response.choices[0].message.content.lower()
    
    # Check if correct category is mentioned
    if true_category in base_pred:
        base_correct += 1
    if true_category in finetuned_pred:
        finetuned_correct += 1
    
    print(f"Example {i+1}:")
    print(f"  True category: {true_category}")
    print(f"  Base model: {'✓' if true_category in base_pred else '✗'}")
    print(f"  Fine-tuned: {'✓' if true_category in finetuned_pred else '✗'}")

print(f"\nResults on {len(test_examples)} examples:")
print(f"Base model accuracy: {base_correct}/{len(test_examples)} ({base_correct/len(test_examples)*100:.0f}%)")
print(f"Fine-tuned accuracy: {finetuned_correct}/{len(test_examples)} ({finetuned_correct/len(test_examples)*100:.0f}%)")

## 10. Summary

Fine-tuning results and next steps.

In [ ]:
print("="*60)
print("FINE-TUNING SUMMARY")
print("="*60)
print(f"\nModel: GPT-4o-mini")
print(f"Fine-tuned model ID: {fine_tuned_model}")
print(f"\nTraining details:")
print(f"  Training examples: {len(training_examples)}")
print(f"  Validation examples: {len(validation_examples)}")
print(f"  Epochs: 3")
print(f"  Batch size: 4")
print(f"  Training time: {model_info.get('training_time_minutes', 'N/A')} minutes")
print(f"\nExpected performance (from paper):")
print(f"  Base model: 67.3% accuracy")
print(f"  Fine-tuned: 83.7% accuracy")
print(f"  Improvement: +16.4%")
print(f"\nNext step: Run full evaluation on test set (notebook 04)")